# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
import json
import chromadb
from chromadb.utils import embedding_functions

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
from dotenv import load_dotenv
load_dotenv(".env")


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("Environment key check:")
print(f"- OPENAI_API_KEY exists: {bool(OPENAI_API_KEY)}")
print(f"- TAVILY_API_KEY exists: {bool(TAVILY_API_KEY)}")

if not OPENAI_API_KEY or not TAVILY_API_KEY:
    print("\nOne or more keys are missing. Check project/starter/.env and rerun the env-loading cell.")

Environment key check:
- OPENAI_API_KEY exists: True
- TAVILY_API_KEY exists: True


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# Build the collection:
# Create the chromadb client and collection, and add all the data from games/ folder to the collection.
import tempfile

db_path = "./chromadb"
os.makedirs(db_path, exist_ok=True)

try:
    chroma_client = chromadb.PersistentClient(path=db_path)
    print(f"Initialized ChromaDB at: {os.path.abspath(db_path)}")
except Exception:
    fallback_db_path = os.path.join(tempfile.gettempdir(), "udaplay_chromadb")
    os.makedirs(fallback_db_path, exist_ok=True)
    chroma_client = chromadb.PersistentClient(path=fallback_db_path)
    print(f"Could not open ./chromadb, switched to: {fallback_db_path}")

chroma_client

embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

collection_name = "udaplay"

try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)
collection

# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

# Retrieve all records from ChromaDB and print each record as JSON
#records = collection.get(include=["documents", "metadatas"])

#for i, record_id in enumerate(records.get("ids", [])):
#    row = {
#        "id": record_id,
#        "document": records["documents"][i] if records.get("documents") else None,
#        "metadata": records["metadatas"][i] if records.get("metadatas") else None,
#   }
#    print(json.dumps(row, ensure_ascii=False, indent=2))


Initialized ChromaDB at: /Users/man.m.do/Documents/GitHub/udacity_build_project/project/starter/chromadb


In [5]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str, n_results: int = 5) -> list[dict]:
    """Semantic search: Finds most results in the vector DB.

    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    result = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["metadatas"]
    )

    rows = (result.get("metadatas") or [[]])[0]

    return [
        {
            "Platform": row.get("Platform"),
            "Name": row.get("Name"),
            "YearOfRelease": row.get("YearOfRelease"),
            "Description": row.get("Description"),
        }
        for row in rows
    ]

# Add invoke() compatibility so this tool can be called as retrieve_game.invoke({...})
if not hasattr(retrieve_game, "invoke"):
    retrieve_game.invoke = lambda payload: retrieve_game(**payload)

sample_query = "When Pokémon Gold and Silver was released?"
retrieved_results = retrieve_game.invoke({"query": sample_query, "n_results": 5})
print(json.dumps(retrieved_results, ensure_ascii=False, indent=2))


[
  {
    "Platform": "Game Boy Color",
    "Name": "Pokémon Gold and Silver",
    "YearOfRelease": 1999,
    "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics."
  },
  {
    "Platform": "Game Boy Advance",
    "Name": "Pokémon Ruby and Sapphire",
    "YearOfRelease": 2002,
    "Description": "Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles."
  },
  {
    "Platform": "Nintendo 64",
    "Name": "Super Mario 64",
    "YearOfRelease": 1996,
    "Description": "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."
  },
  {
    "Platform": "Super Nintendo Entertainment System (SNES)",
    "Name": "Super Mario World",
    "YearOfRelease": 1990,
    "Description": "A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser."
  },
  {
    "Platform": "GameCube",
    "Name": "Su

#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

import json
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    description: str = Field(description="Detailed explanation about retrieval quality and next action")

llm_judge = LLM(model="gpt-4o-mini", temperature=0.0)
report_parser = PydanticOutputParser(model_class=EvaluationReport)

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge_prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Question: {question}

Retrieved documents:
{json.dumps(retrieved_docs, ensure_ascii=False, indent=2)}

Return JSON with:
- useful: boolean
- description: detailed explanation
"""

    ai_message = llm_judge.invoke(
        input=judge_prompt,
        response_format=EvaluationReport
    )
    report = report_parser.parse(ai_message)
    print("Parsed EvaluationReport:")
    print(report.model_dump_json(ensure_ascii=False, indent=2))
    return report.model_dump()



#### Game Web Search Tool

In [7]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient

@tool
def game_web_search(question: str) -> list[dict]:
    """Semantic search: Finds most results in the vector DB.

    args:
    - question: a question about game industry.
    """
    api_key = TAVILY_API_KEY or os.getenv("TAVILY_API_KEY")
    if not api_key:
        return [{
            "title": "Missing API key",
            "url": "",
            "content": "TAVILY_API_KEY is not set."
        }]

    client = TavilyClient(api_key=api_key)
    response = client.search(
        query=question,
        max_results=5,
        search_depth="advanced",
        include_answer=True
    )

    return [
        {
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "content": item.get("content", "")
        }
        for item in response.get("results", [])
    ]

# Quick test
web_query = "latest trends in cloud gaming market 2025"
web_results = game_web_search(question=web_query)
print(json.dumps(web_results, ensure_ascii=False, indent=2))


[
  {
    "title": "Cloud Gaming Market Size & Share | Industry Report, 2030",
    "url": "https://www.grandviewresearch.com/industry-analysis/cloud-gaming-market",
    "content": "### Europe Cloud Gaming Market Trends\n\nThe cloud gaming market in Europe is expected to grow at a CAGR of over 43% from 2025 to 2030.   Increased internet speeds, a strong gaming culture, and a growing appetite for immersive gaming experiences drive this growth. Subscription-based services are gaining popularity, offering cost-effective access to a wide range of games. In addition, the integration of cloud gaming with other entertainment platforms is enhancing user engagement and expanding the reach of services. [...] Grand View Research Logo\nicon\nCloud Gaming Market Size, Share & Trends Report\n\n# Cloud Gaming Market (2025 - 2030) Size, Share & Trends Analysis Report By Type (File Streaming, Video Streaming), By Device (Smartphones, Tablets, Smart TVs), By Gamer Type, By Region, And Segment Forecasts\n

### Agent

In [8]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agent_instructions = """You are UdaPlay, an AI research agent for the video game industry.
Use tools deliberately and in this order:
1) Use retrieve_game first for internal vector DB evidence.
2) Use evaluate_retrieval to judge if retrieved documents are sufficient.
3) If retrieval is insufficient, use game_web_search for fresh external context.
4) Provide a concise and accurate final answer grounded in tool outputs.
"""

tool_map = {
    "retrieve_game": globals().get("retrieve_game"),
    "evaluate_retrieval": globals().get("evaluate_retrieval"),
    "game_web_search": globals().get("game_web_search"),
}
missing_tools = [name for name, tool_obj in tool_map.items() if tool_obj is None]
if missing_tools:
    raise ValueError(
        f"Missing tool definitions: {missing_tools}. Run previous tool cells first."
    )

game_search_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=agent_instructions,
    tools=[
        tool_map["retrieve_game"],
        tool_map["evaluate_retrieval"],
        tool_map["game_web_search"],
    ],
    temperature=0.2,
)

game_search_agent

In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

if "game_search_agent" not in globals():
    raise ValueError("game_search_agent is not defined. Run Cell 17 first.")

test_queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

session_id = "udaplay_validation"
if session_id in game_search_agent.memory.sessions:
    game_search_agent.reset_session(session_id)

def _extract_role(msg):
    return getattr(msg, "role", msg.get("role") if isinstance(msg, dict) else None)

def _extract_content(msg):
    return getattr(msg, "content", msg.get("content") if isinstance(msg, dict) else "")

for idx, query in enumerate(test_queries, start=1):
    run = game_search_agent.invoke(query, session_id=session_id)
    final_state = run.get_final_state() or {}
    messages = final_state.get("messages", [])

    assistant_messages = [
        m for m in messages
        if _extract_role(m) == "assistant" and _extract_content(m)
    ]
    final_answer = _extract_content(assistant_messages[-1]) if assistant_messages else "(No assistant answer found)"

    print(f"\n=== Test {idx} ===")
    print(f"Query: {query}")
    print(f"Answer: {final_answer}")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Parsed EvaluationReport:
{
  "useful": true,
  "description": "The retrieved document provides the name of the game, 'Pokémon Gold and Silver', along with its release year, which is 1999. This directly answers the query regarding when Pokémon Gold and Silver was released. The additional information about the platform (Game Boy Color) and a brief description of the game adds context but is not necessary to answer the specific question. Since the key information needed to respond to the query is present and accurate, the document is deemed useful."
}
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Test 1 ===
Query: When Pokémon Gold and Silver was released?
Answer: Pokémon Gold and Silver

### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

from typing import TypedDict, Optional
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

class AdvancedAgentState(TypedDict):
    user_query: str
    user_id: str
    memory_hits: list[str]
    retrieved_docs: list[dict]
    retrieval_eval: dict
    web_results: list[dict]
    final_answer: str

class GameSearchAgentWithLongTermMemory:
    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0.2):
        self.llm = LLM(model=model_name, temperature=temperature)

        if not OPENAI_API_KEY:
            raise ValueError("OPENAI_API_KEY is required to initialize long-term memory.")

        self.vector_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
        self.long_term_memory = LongTermMemory(self.vector_manager)
        self.workflow = self._create_workflow()

    def _memory_recall_step(self, state: AdvancedAgentState) -> dict:
        try:
            memory_result = self.long_term_memory.search(
                query_text=state["user_query"],
                owner=state["user_id"],
                limit=3,
                namespace="udaplay",
            )
            memory_hits = [fragment.content for fragment in memory_result.fragments]
        except Exception:
            memory_hits = []

        return {"memory_hits": memory_hits}

    def _retrieve_step(self, state: AdvancedAgentState) -> dict:
        docs = retrieve_game(query=state["user_query"], n_results=5)
        return {"retrieved_docs": docs}

    def _evaluate_step(self, state: AdvancedAgentState) -> dict:
        evaluation = evaluate_retrieval(
            question=state["user_query"],
            retrieved_docs=state.get("retrieved_docs", []),
        )
        return {"retrieval_eval": evaluation}

    def _web_search_step(self, state: AdvancedAgentState) -> dict:
        web_results = game_web_search(question=state["user_query"])
        return {"web_results": web_results}

    def _compose_answer_step(self, state: AdvancedAgentState) -> dict:
        prompt = f"""You are UdaPlay, a video-game research assistant.

User question: {state['user_query']}

Relevant long-term memory (if any):
{json.dumps(state.get('memory_hits', []), ensure_ascii=False, indent=2)}

Retrieved vector DB docs:
{json.dumps(state.get('retrieved_docs', []), ensure_ascii=False, indent=2)}

Retrieval evaluation:
{json.dumps(state.get('retrieval_eval', {}), ensure_ascii=False, indent=2)}

Web search results (if any):
{json.dumps(state.get('web_results', []), ensure_ascii=False, indent=2)}

Write a concise final answer grounded in the evidence above.
If evidence is weak, clearly say uncertainty.
"""

        ai_message = self.llm.invoke(prompt)
        return {"final_answer": ai_message.content or ""}

    def _save_memory_step(self, state: AdvancedAgentState) -> dict:
        answer = state.get("final_answer", "").strip()
        if answer:
            memory_text = f"Q: {state['user_query']}\nA: {answer}"
            fragment = MemoryFragment(
                content=memory_text,
                owner=state["user_id"],
                namespace="udaplay",
            )
            self.long_term_memory.register(fragment)
        return {}

    def _route_after_evaluation(self, state: AdvancedAgentState):
        useful = bool(state.get("retrieval_eval", {}).get("useful", False))
        if useful:
            return "compose_answer"
        return "web_search"

    def _create_workflow(self) -> StateMachine[AdvancedAgentState]:
        machine = StateMachine[AdvancedAgentState](AdvancedAgentState)

        entry = EntryPoint[AdvancedAgentState]()
        memory_recall = Step[AdvancedAgentState]("memory_recall", self._memory_recall_step)
        retrieve = Step[AdvancedAgentState]("retrieve_game", self._retrieve_step)
        evaluate = Step[AdvancedAgentState]("evaluate_retrieval", self._evaluate_step)
        web_search = Step[AdvancedAgentState]("web_search", self._web_search_step)
        compose = Step[AdvancedAgentState]("compose_answer", self._compose_answer_step)
        save_memory = Step[AdvancedAgentState]("save_memory", self._save_memory_step)
        termination = Termination[AdvancedAgentState]()

        machine.add_steps([
            entry, memory_recall, retrieve, evaluate, web_search, compose, save_memory, termination
        ])

        machine.connect(entry, memory_recall)
        machine.connect(memory_recall, retrieve)
        machine.connect(retrieve, evaluate)
        machine.connect(evaluate, [web_search, compose], self._route_after_evaluation)
        machine.connect(web_search, compose)
        machine.connect(compose, save_memory)
        machine.connect(save_memory, termination)

        return machine

    def invoke(self, query: str, user_id: str = "default") -> Run:
        initial_state: AdvancedAgentState = {
            "user_query": query,
            "user_id": user_id,
            "memory_hits": [],
            "retrieved_docs": [],
            "retrieval_eval": {},
            "web_results": [],
            "final_answer": "",
        }
        return self.workflow.run(initial_state)

advanced_game_agent = GameSearchAgentWithLongTermMemory()
advanced_game_agent

In [11]:
# Validate long-term memory + state-machine workflow
if "advanced_game_agent" not in globals():
    raise ValueError("advanced_game_agent is not defined. Run Cell 20 first.")

user_id = "udaplay_user_01"
test_question = "When Pokémon Gold and Silver was released?"

print("=== Run 1 (should create memory) ===")
run1 = advanced_game_agent.invoke(test_question, user_id=user_id)
state1 = run1.get_final_state() or {}
steps1 = [snapshot.step_id for snapshot in run1.snapshots]
print("State-machine steps:", steps1)
print("Final answer:", state1.get("final_answer", ""))

print("\n=== Run 2 (should recall memory) ===")
run2 = advanced_game_agent.invoke(test_question, user_id=user_id)
state2 = run2.get_final_state() or {}
steps2 = [snapshot.step_id for snapshot in run2.snapshots]
print("State-machine steps:", steps2)
print("Memory hits count:", len(state2.get("memory_hits", [])))
print("Memory hits preview:", state2.get("memory_hits", [])[:1])
print("Final answer:", state2.get("final_answer", ""))

print("\n=== Validation checks ===")
workflow_ok = all(step in steps2 for step in ["memory_recall", "retrieve_game", "evaluate_retrieval", "compose_answer", "save_memory"] )
memory_ok = len(state2.get("memory_hits", [])) > 0
print("Workflow node sequence used:", workflow_ok)
print("Long-term memory recalled:", memory_ok)

=== Run 1 (should create memory) ===
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: retrieve_game
Parsed EvaluationReport:
{
  "useful": true,
  "description": "The retrieved documents include a specific entry for 'Pokémon Gold and Silver' which states that it was released in 1999. This directly answers the query regarding the release date of the game. Although there are other entries for different games, they do not detract from the relevance of the information provided about Pokémon Gold and Silver. Therefore, the documents are sufficient to respond to the question."
}
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: compose_answer
[StateMachine] Executing step: save_memory
[StateMachine] Terminating: __termination__
State-machine steps: ['__entry__', 'memory_recall', 'retrieve_game', 'evaluate_retrieval', 'compose_answer', 'save_memory']
Final answer: Pokémon Gold and Silver was released 

### Retrieval Process Dashboard

In [12]:
# Dashboard / visualization for the agent retrieval process (dependency-free)
from collections import Counter
from html import escape
from IPython.display import display, HTML

if "advanced_game_agent" not in globals():
    raise ValueError("advanced_game_agent is not defined. Run the Advanced agent cell first.")

def summarize_run(run_obj, query_text: str) -> dict:
    snapshots = getattr(run_obj, "snapshots", [])
    step_sequence = [snapshot.step_id for snapshot in snapshots]
    final_state = run_obj.get_final_state() or {}

    retrieved_docs = final_state.get("retrieved_docs", []) or []
    memory_hits = final_state.get("memory_hits", []) or []
    web_results = final_state.get("web_results", []) or []
    eval_result = final_state.get("retrieval_eval", {}) or {}

    return {
        "query": query_text,
        "step_sequence": step_sequence,
        "steps_count": len(step_sequence),
        "retrieved_docs_count": len(retrieved_docs),
        "memory_hits_count": len(memory_hits),
        "web_results_count": len(web_results),
        "retrieval_useful": bool(eval_result.get("useful", False)),
        "web_fallback_used": "web_search" in step_sequence,
    }

def bar(width_percent: float, color: str) -> str:
    width_percent = max(0.0, min(100.0, width_percent))
    return (
        f"<div style='background:#f1f5f9;border-radius:6px;height:14px;width:160px;'>"
        f"<div style='height:14px;border-radius:6px;background:{color};width:{width_percent:.1f}%;'></div>"
        "</div>"
    )

dashboard_user_id = "udaplay_dashboard_user"
dashboard_queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
    "What are 2026 trends in cloud gaming adoption?",
]

run_summaries = []
for q in dashboard_queries:
    run_obj = advanced_game_agent.invoke(q, user_id=dashboard_user_id)
    run_summaries.append(summarize_run(run_obj, q))

# Aggregate node usage for process visibility
all_steps = [step for item in run_summaries for step in item["step_sequence"]]
step_counts = Counter(all_steps)
max_step_count = max(step_counts.values()) if step_counts else 1

step_rows = []
for step_name, count in step_counts.items():
    width = (count / max_step_count) * 100
    step_rows.append(
        f"<tr>"
        f"<td style='padding:6px 8px;border-bottom:1px solid #e2e8f0;'>{escape(step_name)}</td>"
        f"<td style='padding:6px 8px;border-bottom:1px solid #e2e8f0;'>{count}</td>"
        f"<td style='padding:6px 8px;border-bottom:1px solid #e2e8f0;'>{bar(width, '#2563eb')}</td>"
        f"</tr>"
    )

summary_rows = []
max_docs = max([item["retrieved_docs_count"] for item in run_summaries] + [1])
max_memory = max([item["memory_hits_count"] for item in run_summaries] + [1])
max_web = max([item["web_results_count"] for item in run_summaries] + [1])

for item in run_summaries:
    docs_pct = (item["retrieved_docs_count"] / max_docs) * 100
    memory_pct = (item["memory_hits_count"] / max_memory) * 100
    web_pct = (item["web_results_count"] / max_web) * 100

    summary_rows.append(
        "<tr>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;max-width:360px;'>{escape(item['query'])}</td>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;'>{item['steps_count']}</td>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;'>{bar(docs_pct, '#16a34a')}<div style='font-size:12px;color:#475569'>{item['retrieved_docs_count']}</div></td>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;'>{bar(memory_pct, '#7c3aed')}<div style='font-size:12px;color:#475569'>{item['memory_hits_count']}</div></td>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;'>{bar(web_pct, '#ea580c')}<div style='font-size:12px;color:#475569'>{item['web_results_count']}</div></td>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;'>{'Yes' if item['retrieval_useful'] else 'No'}</td>"
        f"<td style='padding:8px;border-bottom:1px solid #e2e8f0;'>{'Yes' if item['web_fallback_used'] else 'No'}</td>"
        "</tr>"
    )

html = f"""
<div style='font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif;'>
  <h3 style='margin-bottom:8px;'>Retrieval Process Dashboard</h3>

  <h4 style='margin:12px 0 6px;'>Node Usage</h4>
  <table style='border-collapse:collapse;min-width:520px;'>
    <thead>
      <tr>
        <th style='text-align:left;padding:6px 8px;border-bottom:2px solid #cbd5e1;'>Step</th>
        <th style='text-align:left;padding:6px 8px;border-bottom:2px solid #cbd5e1;'>Count</th>
        <th style='text-align:left;padding:6px 8px;border-bottom:2px solid #cbd5e1;'>Relative Usage</th>
      </tr>
    </thead>
    <tbody>
      {''.join(step_rows)}
    </tbody>
  </table>

  <h4 style='margin:16px 0 6px;'>Per-Query Retrieval Outcome</h4>
  <table style='border-collapse:collapse;min-width:1000px;'>
    <thead>
      <tr>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Query</th>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Steps</th>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Retrieved Docs</th>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Memory Hits</th>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Web Results</th>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Retrieval Useful?</th>
        <th style='text-align:left;padding:8px;border-bottom:2px solid #cbd5e1;'>Web Fallback?</th>
      </tr>
    </thead>
    <tbody>
      {''.join(summary_rows)}
    </tbody>
  </table>
</div>
"""

display(HTML(html))

# Step-by-step trace remains useful for debugging agent routes
for i, item in enumerate(run_summaries, start=1):
    print(f"\n=== Query {i} Trace ===")
    print("Query:", item["query"])
    print("Step sequence:", " -> ".join(item["step_sequence"]))
    print(
        "retrieved_docs=", item["retrieved_docs_count"],
        "| memory_hits=", item["memory_hits_count"],
        "| web_results=", item["web_results_count"],
        "| retrieval_useful=", item["retrieval_useful"],
    )

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: retrieve_game
Parsed EvaluationReport:
{
  "useful": true,
  "description": "The retrieved documents include a specific entry for 'Pokémon Gold and Silver' which states that it was released in 1999. This directly answers the query regarding the release date of the game. Although there are other entries for different games, they do not detract from the relevance of the information provided about Pokémon Gold and Silver. Therefore, the documents are sufficient to respond to the question."
}
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: compose_answer
[StateMachine] Executing step: save_memory
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: retrieve_game
Parsed EvaluationReport:
{
  "useful": true,
  "description": "The retrieved documents 


=== Query 1 Trace ===
Query: When Pokémon Gold and Silver was released?
Step sequence: __entry__ -> memory_recall -> retrieve_game -> evaluate_retrieval -> compose_answer -> save_memory
retrieved_docs= 5 | memory_hits= 0 | web_results= 0 | retrieval_useful= True

=== Query 2 Trace ===
Query: Which one was the first 3D platformer Mario game?
Step sequence: __entry__ -> memory_recall -> retrieve_game -> evaluate_retrieval -> compose_answer -> save_memory
retrieved_docs= 5 | memory_hits= 1 | web_results= 0 | retrieval_useful= True

=== Query 3 Trace ===
Query: Was Mortal Kombat X released for PlayStation 5?
Step sequence: __entry__ -> memory_recall -> retrieve_game -> evaluate_retrieval -> web_search -> compose_answer -> save_memory
retrieved_docs= 5 | memory_hits= 2 | web_results= 5 | retrieval_useful= False

=== Query 4 Trace ===
Query: What are 2026 trends in cloud gaming adoption?
Step sequence: __entry__ -> memory_recall -> retrieve_game -> evaluate_retrieval -> web_search -> compos